<a href="https://colab.research.google.com/github/DeviShree2006/ITA0620/blob/main/Decision_tree_for_play_tennis_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
"""
Decision Tree (ID3-style, entropy / information gain) on a 14-row real-world dataset.
Part 1: Entropy & Information Gain calculated manually (shows how the algorithm picks the root).
Part 2: Full tree built with scikit-learn, printed as text rules, drawn as a diagram,
        and used to predict a new day.
"""
import math
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

# ---------------------------------------------------------------
# 1. Dataset: 14 days of weather and whether tennis was played
# ---------------------------------------------------------------
data = {
    "Outlook":     ["Sunny", "Sunny", "Overcast", "Rain", "Rain", "Rain", "Overcast",
                    "Sunny", "Sunny", "Rain", "Sunny", "Overcast", "Overcast", "Rain"],
    "Temperature": ["Hot", "Hot", "Hot", "Mild", "Cool", "Cool", "Cool",
                    "Mild", "Cool", "Mild", "Mild", "Mild", "Hot", "Mild"],
    "Humidity":    ["High", "High", "High", "High", "Normal", "Normal", "Normal",
                    "High", "Normal", "Normal", "Normal", "High", "Normal", "High"],
    "Wind":        ["Weak", "Strong", "Weak", "Weak", "Weak", "Strong", "Strong",
                    "Weak", "Weak", "Weak", "Strong", "Strong", "Weak", "Strong"],
    "PlayTennis":  ["No", "No", "Yes", "Yes", "Yes", "No", "Yes",
                    "No", "Yes", "Yes", "Yes", "Yes", "Yes", "No"],
}
df = pd.DataFrame(data)
print("Dataset (14 rows):\n", df, "\n")

# ---------------------------------------------------------------
# 2. Manual Entropy and Information Gain
# ---------------------------------------------------------------
def entropy(series):
    probs = series.value_counts(normalize=True)
    return -sum(p * math.log2(p) for p in probs if p > 0)

def information_gain(df, feature, target="PlayTennis"):
    total_entropy = entropy(df[target])
    weighted = 0
    for value, subset in df.groupby(feature):
        weighted += (len(subset) / len(df)) * entropy(subset[target])
    return total_entropy - weighted

print(f"Entropy of target (PlayTennis): {entropy(df['PlayTennis']):.4f}")
print("\nInformation Gain of each feature:")
gains = {f: information_gain(df, f) for f in ["Outlook", "Temperature", "Humidity", "Wind"]}
for f, g in sorted(gains.items(), key=lambda x: -x[1]):
    print(f"  {f:<12} {g:.4f}")
print(f"\n=> Root node = '{max(gains, key=gains.get)}' (highest information gain)\n")

# ---------------------------------------------------------------
# 3. Build the tree with scikit-learn (one-hot encoding -> readable rules)
# ---------------------------------------------------------------
X = pd.get_dummies(df.drop("PlayTennis", axis=1)).astype(int)
y = df["PlayTennis"]

model = DecisionTreeClassifier(criterion="entropy", random_state=42)
model.fit(X, y)

print("Tree rules (value <= 0.5 means 'No', > 0.5 means 'Yes'):")
print(export_text(model, feature_names=list(X.columns)))
print(f"Training accuracy: {model.score(X, y):.2f}\n")

# ---------------------------------------------------------------
# 4. Predict a new day: Sunny, Cool, High humidity, Strong wind
# ---------------------------------------------------------------
new_day = pd.DataFrame([{"Outlook": "Sunny", "Temperature": "Cool",
                         "Humidity": "High", "Wind": "Strong"}])
new_X = pd.get_dummies(new_day).reindex(columns=X.columns, fill_value=0).astype(int)
print("New day:", new_day.iloc[0].to_dict())
print("Prediction -> PlayTennis =", model.predict(new_X)[0])

# ---------------------------------------------------------------
# 5. Draw the tree structure and save it as an image
# ---------------------------------------------------------------
plt.figure(figsize=(20, 10))
plot_tree(model,
          feature_names=list(X.columns),
          class_names=list(model.classes_),
          filled=True, rounded=True, fontsize=10)
plt.title("Decision Tree - Play Tennis (criterion = entropy)", fontsize=16)
plt.tight_layout()
plt.savefig("decision_tree.png", dpi=150)
print("Tree diagram saved as decision_tree.png")
plt.show()

Dataset (14 rows):
      Outlook Temperature Humidity    Wind PlayTennis
0      Sunny         Hot     High    Weak         No
1      Sunny         Hot     High  Strong         No
2   Overcast         Hot     High    Weak        Yes
3       Rain        Mild     High    Weak        Yes
4       Rain        Cool   Normal    Weak        Yes
5       Rain        Cool   Normal  Strong         No
6   Overcast        Cool   Normal  Strong        Yes
7      Sunny        Mild     High    Weak         No
8      Sunny        Cool   Normal    Weak        Yes
9       Rain        Mild   Normal    Weak        Yes
10     Sunny        Mild   Normal  Strong        Yes
11  Overcast        Mild     High  Strong        Yes
12  Overcast         Hot   Normal    Weak        Yes
13      Rain        Mild     High  Strong         No 

Entropy of target (PlayTennis): 0.9403

Information Gain of each feature:
  Outlook      0.2467
  Humidity     0.1518
  Wind         0.0481
  Temperature  0.0292

=> Root node = 'Outl